# 自定义中间件
通过实现 hooks（钩子），在 Agent 执行流程的特定节点运行自定义逻辑，从而构建自定义中间件。

中间件提供 两种 hook 风格，用于拦截 Agent 的执行流程。
## Node-style hooks（节点式钩子）
在特定执行节点 按顺序运行。

适用于日志记录、输入校验、Agent 状态更新等场景。

可用 hooks

before_agent：Agent 启动前（每次调用仅一次）

before_model：每次模型调用前

after_model：每次模型返回后

after_agent：Agent 执行结束后（每次调用仅一次）

装饰器方式

In [ ]:
from langchain.agents.middleware import before_model, after_model, AgentState
from langchain.messages import AIMessage
from langgraph.runtime import Runtime
from typing import Any

@before_model(can_jump_to=["end"])
def check_message_limit(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    if len(state["messages"]) >= 50:
        return {
            "messages": [AIMessage("Conversation limit reached.")],
            "jump_to": "end"
        }
    return None

@after_model
def log_response(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    print(f"Model returned: {state['messages'][-1].content}")
    return None

类方式

In [ ]:
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langchain.messages import AIMessage
from langgraph.runtime import Runtime
from typing import Any

class MessageLimitMiddleware(AgentMiddleware):
    def __init__(self, max_messages: int = 50):
        super().__init__()
        self.max_messages = max_messages

    @hook_config(can_jump_to=["end"])
    def before_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if len(state["messages"]) == self.max_messages:
            return {
                "messages": [AIMessage("Conversation limit reached.")],
                "jump_to": "end"
            }
        return None

    def after_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        print(f"Model returned: {state['messages'][-1].content}")
        return None

## Wrap-style hooks（包装式钩子）
拦截执行流程，并由你控制 handler 何时、是否、以及调用多少次。

可以选择：

0 次：短路执行（直接返回）

1 次：正常流程

多次：实现重试逻辑

可用 hooks

wrap_model_call：包装每一次模型调用

wrap_tool_call：包装每一次工具调用

装饰器方式：


In [ ]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def retry_model(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    for attempt in range(3):
        try:
            return handler(request)
        except Exception as e:
            if attempt == 2:
                raise
            print(f"Retry {attempt + 1}/3 after error: {e}")

类方式

In [ ]:
from langchain.agents.middleware import AgentMiddleware, ModelRequest, ModelResponse
from typing import Callable

class RetryMiddleware(AgentMiddleware):
    def __init__(self, max_retries: int = 3):
        super().__init__()
        self.max_retries = max_retries

    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse:
        for attempt in range(self.max_retries):
            try:
                return handler(request)
            except Exception as e:
                if attempt == self.max_retries - 1:
                    raise
                print(f"Retry {attempt + 1}/{self.max_retries} after error: {e}")

## 执行顺序
在 LangChain 中，wrap-style hooks 中间件本质上是一个 “洋葱模型（onion model）” 的调用链。每个 hook 会在调用前后包裹下一层 hook 或核心逻辑，因此执行顺序呈现 前进时顺序执行，返回时逆序执行。